# Missing data and class imbalance

Exercise notebook for the lesson **[Missing data and class imbalance](https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance)** in the *Machine learning* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** In clinical data, whether a value is missing often depends on how ill the patient seemed, so dropping incomplete records or ignoring missingness can bias a model. For prediction, imputation inside the modelling pipeline, sometimes with a 'was it measured?' flag, usually works best, as long as testing practices stay stable. Rare outcomes rarely need 'correcting': oversampling or reweighting leaves AUROC unchanged and makes predicted risks far too high.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Informative missingness, in code

Synthetic patients in whom lactate is measured mainly when severity is high.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(13)
n = 8000
severity = rng.normal(0, 1, n)
lactate = (1.5 + 0.8 * severity + rng.normal(0, 0.5, n)).clip(0.3).round(1)
died = rng.binomial(1, 1 / (1 + np.exp(-(-3.0 + 1.6 * severity))))
measured = rng.binomial(1, 1 / (1 + np.exp(-(-0.5 + 1.8 * severity)))).astype(bool)
df = pd.DataFrame({"age": rng.integers(18, 90, n),
                   "heart_rate": (85 + 6 * severity + rng.normal(0, 12, n)).round(),
                   "lactate": np.where(measured, lactate, np.nan),
                   "died": died})
print(f"Lactate missing in {df.lactate.isna().mean():.0%} of patients")
print(df.groupby(df.lactate.isna()).died.mean().rename("death rate").round(3))

# Lactate missing in 58% of patients
# lactate
# False    0.184
# True     0.042
# Name: death rate, dtype: float64

### Dropping, imputing and flagging

Continuing from the previous block: three ways to handle missing lactate, evaluated on the same test set.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = df[["age", "heart_rate", "lactate"]], df.died
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0, stratify=y)

def evaluate(columns, imputer, label):
    model = make_pipeline(imputer, StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_train[columns], y_train)
    p = model.predict_proba(X_test[columns])[:, 1]
    print(f"{label:<34} AUROC {roc_auc_score(y_test, p):.3f}")

evaluate(["age", "heart_rate"], SimpleImputer(), "Lactate dropped")
evaluate(["age", "heart_rate", "lactate"], SimpleImputer(strategy="median"),
         "Median imputation")
evaluate(["age", "heart_rate", "lactate"],
         SimpleImputer(strategy="median", add_indicator=True),
         "Median imputation + missing flag")

# Lactate dropped                    AUROC 0.658
# Median imputation                  AUROC 0.768
# Median imputation + missing flag   AUROC 0.784

### Class imbalance: usually not a problem to fix

A rare outcome (about 5%) with and without class reweighting.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(4)
n = 20000
X = rng.normal(size=(n, 4))
y = rng.binomial(1, 1 / (1 + np.exp(-(-3.5 + X @ np.array([0.9, 0.6, 0.4, 0.0])))))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=0, stratify=y)
print(f"Outcome rate {y.mean():.1%}")

for label, weights in [("No correction", None), ("class_weight='balanced'", "balanced")]:
    model = LogisticRegression(class_weight=weights).fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    print(f"{label:<24} AUROC {roc_auc_score(y_test, p):.3f}   "
          f"mean predicted risk {p.mean():.1%}   observed {y_test.mean():.1%}")

# Outcome rate 4.7%
# No correction            AUROC 0.811   mean predicted risk 4.7%   observed 4.7%
# class_weight='balanced'  AUROC 0.811   mean predicted risk 37.4%   observed 4.7%

## Exercise

**Goal:** Handle informative missingness and a rare outcome properly, and measure the consequences of common shortcuts.

**Data:** Synthetic patients generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the lactate example and add a complete-case model; compare its mean predicted risk with the observed rate in the full test set.

In [ ]:
# Your code here


### Step 2

Replace median imputation with IterativeImputer and compare AUROC and calibration.

In [ ]:
# Your code here


### Step 3

Simulate a change in practice in which lactate is measured in everyone, and evaluate the model with the missingness flag on this new data.

In [ ]:
# Your code here


### Step 4

Reproduce the imbalance example, then try random oversampling and SMOTE (imbalanced-learn); compare AUROC, calibration plots and Brier scores.

In [ ]:
# Your code here


### Step 5

For the uncorrected model, choose a threshold that gives 80% sensitivity and report the resulting PPV.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance](https://ramkumarmd.com/medical-ai/machine-learning/missing-data-and-imbalance). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.